# Lexical Diversity Analysis

This notebook evaluates lexical diversity between original and synthetic datasets using:

1. **Type-Token Ratio (TTR)** - Measures vocabulary variation within a dataset
2. **Jaccard Index** - Quantifies lexical overlap between two datasets


## 1. Setup

In [1]:
%pip install -q pandas transformers


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from typing import List, Set
from transformers import AutoTokenizer

# Load CafeBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("uitnlp/CafeBERT")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Lexical Diversity Functions

In [3]:
def get_tokens(texts: List[str]) -> List[str]:
    """Tokenize texts using CafeBERT tokenizer."""
    tokens = []
    for text in texts:
        if pd.isna(text):
            continue
        words = tokenizer.tokenize(str(text).lower())
        tokens.extend(words)
    return tokens

def get_types(tokens: List[str]) -> Set[str]:
    """Get unique word types from tokens."""
    return set(tokens)

def compute_ttr(texts: List[str]) -> dict:
    """
    Compute Type-Token Ratio (TTR).
    TTR = Number of Unique Words (Types) / Total Number of Words (Tokens)
    
    Returns:
        dict with types, tokens, and TTR value
    """
    tokens = get_tokens(texts)
    types = get_types(tokens)
    
    num_tokens = len(tokens)
    num_types = len(types)
    ttr = num_types / num_tokens if num_tokens > 0 else 0
    
    return {
        'types': num_types,
        'tokens': num_tokens,
        'ttr': ttr
    }

def compute_jaccard(texts_a: List[str], texts_b: List[str]) -> dict:
    """
    Compute Jaccard Index between two datasets.
    J(A, B) = |A ∩ B| / |A ∪ B|
    
    Args:
        texts_a: Original dataset texts
        texts_b: Synthetic dataset texts
        
    Returns:
        dict with intersection, union sizes and Jaccard index
    """
    types_a = get_types(get_tokens(texts_a))
    types_b = get_types(get_tokens(texts_b))
    
    intersection = types_a & types_b
    union = types_a | types_b
    
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0
    
    return {
        'types_a': len(types_a),
        'types_b': len(types_b),
        'intersection': len(intersection),
        'union': len(union),
        'jaccard': jaccard
    }

## 3. Load Data

In [4]:
# Load original and synthetic datasets
original_path = '/Users/giahuy/Documents/GitHub/topicmodeling/tm_research/data/processed/train_processed.csv'
synthetic_path = '/Users/giahuy/Documents/GitHub/topicmodeling/tm_research/data/processed/train_1300_final.csv'

original_df = pd.read_csv(original_path)
synthetic_df = pd.read_csv(synthetic_path)

print(f"Original dataset: {len(original_df)} samples")
print(f"Synthetic dataset: {len(synthetic_df)} samples")

Original dataset: 5548 samples
Synthetic dataset: 17491 samples


## 4. Compute Lexical Diversity Metrics

In [5]:
# Extract texts
text_column = 'Sentence_clean'
original_texts = original_df[text_column].tolist()
synthetic_texts = synthetic_df[text_column].tolist()

# Compute TTR for both datasets
print("Computing TTR...")
ttr_original = compute_ttr(original_texts)
ttr_synthetic = compute_ttr(synthetic_texts)

print("\n=== Type-Token Ratio (TTR) ===")
print(f"\nOriginal Dataset:")
print(f"  Types (unique words): {ttr_original['types']:,}")
print(f"  Tokens (total words): {ttr_original['tokens']:,}")
print(f"  TTR: {ttr_original['ttr']:.4f}")

print(f"\nSynthetic Dataset:")
print(f"  Types (unique words): {ttr_synthetic['types']:,}")
print(f"  Tokens (total words): {ttr_synthetic['tokens']:,}")
print(f"  TTR: {ttr_synthetic['ttr']:.4f}")

Computing TTR...

=== Type-Token Ratio (TTR) ===

Original Dataset:
  Types (unique words): 2,651
  Tokens (total words): 91,477
  TTR: 0.0290

Synthetic Dataset:
  Types (unique words): 2,954
  Tokens (total words): 330,581
  TTR: 0.0089


In [6]:
# Compute Jaccard Index
print("\nComputing Jaccard Index...")
jaccard_result = compute_jaccard(original_texts, synthetic_texts)

print("\n=== Jaccard Index ===")
print(f"\nOriginal types: {jaccard_result['types_a']:,}")
print(f"Synthetic types: {jaccard_result['types_b']:,}")
print(f"Intersection (shared vocabulary): {jaccard_result['intersection']:,}")
print(f"Union (combined vocabulary): {jaccard_result['union']:,}")
print(f"Jaccard Index: {jaccard_result['jaccard']:.4f}")


Computing Jaccard Index...

=== Jaccard Index ===

Original types: 2,651
Synthetic types: 2,954
Intersection (shared vocabulary): 2,651
Union (combined vocabulary): 2,954
Jaccard Index: 0.8974


In [7]:
# Summary table
print("\n=== Summary ===")
summary_data = {
    'Metric': ['Types', 'Tokens', 'TTR', 'Jaccard Index'],
    'Original': [ttr_original['types'], ttr_original['tokens'], f"{ttr_original['ttr']:.4f}", '-'],
    'Synthetic': [ttr_synthetic['types'], ttr_synthetic['tokens'], f"{ttr_synthetic['ttr']:.4f}", '-'],
    'Comparison': ['-', '-', '-', f"{jaccard_result['jaccard']:.4f}"]
}
pd.DataFrame(summary_data)


=== Summary ===


,Metric,Original,Synthetic,Comparison
0,Types,2651,2954,-
1,Tokens,91477,330581,-
2,TTR,0.0290,0.0089,-
3,Jaccard Index,-,-,0.8974
